# imports

In [1]:
import os
import random
import numpy as np
import torch
import torchaudio
import librosa
from pathlib import Path
from datasets import load_from_disk, Dataset, DatasetDict, Audio
from typing import Optional
from tqdm.auto import tqdm


/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

In [2]:
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH       = PROJECT_ROOT / 'data' / 'synthetic' / 'paired'
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / 'data' / 'synthetic' / 'audio'
TEANGLANN_AUDIO_DIR = PROJECT_ROOT / 'data' / 'teanglann' / 'wav_files'
OUTPUT_PATH        = PROJECT_ROOT / 'data' / 'synthetic' / 'l2_training_set'


In [3]:
paired_ds = load_from_disk(str(DATASET_PATH))
print(paired_ds)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions', 'synthetic_audio_path'],
    num_rows: 19136
})


# Inspect Data

In [4]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾˠ ə bˠ ə d̪ˠ',
 'English ASR transcriptions': 'ʌɾəbʌd',
 'synthetic_audio_path': {'path': '0.mp3',
  'array': array([ 3.18323146e-12,  2.27373675e-12, -9.54969437e-12, ...,
          8.29194323e-06, -2.34242179e-06, -2.22762756e-05], shape=(11136,)),
  'sampling_rate': 16000}}

Not great that I am saving audio inconsistently, but so long as the final dataset has the audio column with the path and array I will keep this as a lesson learned for next time.

# Prepare Data

## strip diacritics

In [5]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.data_handling.normalize_phonemes import normalize_row

In [6]:
paired_ds = paired_ds.map(
    lambda row: normalize_row(row, 'phonetic'),
    load_from_cache_file=False,
    desc="Normalizing phonetic column"
)

Normalizing phonetic column: 100%|██████████| 19136/19136 [00:02<00:00, 8983.35 examples/s] 


## make paired set

In [7]:
TARGET_SR = 16_000
IPA_SEPARATOR = " "
RANDOM_SEED = 42


def _to_mono_16k(audio_array: np.ndarray, source_sr: int) -> np.ndarray:
    """Resample and convert to mono float32 at TARGET_SR."""
    waveform = torch.from_numpy(audio_array.astype(np.float32))
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if source_sr != TARGET_SR:
        resampler = torchaudio.transforms.Resample(orig_freq=source_sr, new_freq=TARGET_SR)
        waveform = resampler(waveform)
    return waveform.squeeze(0).numpy()


def _make_l2_pair(item: dict, idx: int) -> dict:
    """Map function: concatenate native + TTS audio for one word."""
    ref_audio = _to_mono_16k(item["audio"]["array"], item["audio"]["sampling_rate"])

    synth_array = item["synthetic_audio_path"]["array"]
    synth_sr = item["synthetic_audio_path"]["sampling_rate"]
    synth_audio = _to_mono_16k(synth_array, synth_sr)

    # Per-item seeding gives the same reproducibility as a shared rng.
    if random.Random(RANDOM_SEED + idx).random() < 0.5: # Vary order so ordering isn't introducing bias
        combined = np.concatenate([ref_audio, synth_audio])
        ipa = item["phonetic"] + IPA_SEPARATOR + item["English ASR transcriptions"]
        order = "ref_first"
    else:
        combined = np.concatenate([synth_audio, ref_audio])
        ipa = item["English ASR transcriptions"] + IPA_SEPARATOR + item["phonetic"]
        order = "synth_first"

    return {"audio": {"array": combined, "sampling_rate": TARGET_SR}, "ipa": ipa, "order": order}


def build_l2_training_set(dataset: Dataset, seed: Optional[int] = RANDOM_SEED) -> Dataset:
    """
    Build an adversarially-concatenated dataset directly from the paired dataset.

    Input columns:  audio, phonetic, synthetic_audio_path, English ASR transcriptions
    Output columns: audio, ipa, order
    """
    dataset = dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    out_ds = dataset.map(
        _make_l2_pair,
        with_indices=True,
        remove_columns=["phonetic", "synthetic_audio_path", "English ASR transcriptions"],
        desc="Building L2 training set",
    )
    return out_ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))


forgot to normalize transcriptions to same wav2vec2-friendly format as the teanglann scrapings. better for spaces between characters to keep diacritics together with their phones

In [8]:
import json

path = '/home/peter/Desktop/thesis/ThesisProject/scripts/data_handling/L2/combine_ds.ipynb'
with open(path) as f:
    nb = json.load(f)

# Insert a new cell between load (id=43375ba5) and build_l2_training_set (id=693de03f)

SYNTH_VOCAB = {' ': 0, 'aɪ': 1, 'aʊ': 2, 'b': 3, 'd': 4, 'eɪ': 5, 'f': 6, 'g': 7,
               'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'l̩': 13, 'm': 14, 'm̩': 15,
               'n': 16, 'n̩': 17, 'oʊ': 18, 'p': 19, 's': 20, 't': 21, 'u': 22, 'v': 23,
               'w': 24, 'z': 25, 'æ': 26, 'ð': 27, 'ŋ': 28, 'ŋ̍': 29, 'ɑ': 30, 'ɔ': 31,
               'ɔɪ': 32, 'ə': 33, 'ə̥': 34, 'ɚ': 35, 'ɛ': 36, 'ɝ': 37, 'ɦ': 38, 'ɨ': 39,
               'ɪ': 40, 'ɹ': 41, 'ɾ': 42, 'ɾ̃': 43, 'ʃ': 44, 'ʉ': 45, 'ʊ': 46, 'ʌ': 47,
               'ʒ': 48, 'ʔ': 49, 'ʤ': 50, 'ʧ': 51, 'θ': 52}

# Partition into 2-codepoint and 1-codepoint sets for greedy matching
_PHONES_2 = {k for k in SYNTH_VOCAB if len(k) == 2 and k != ' '}
_PHONES_1 = {k for k in SYNTH_VOCAB if len(k) == 1 and k != ' '}


def _space_separate(ipa: str) -> str:
    """Insert spaces between phones in a collapsed synthetic IPA string."""
    if ' ' in ipa:          # already separated
        return ipa
    phones, i = [], 0
    while i < len(ipa):
        if ipa[i:i+2] in _PHONES_2:
            phones.append(ipa[i:i+2])
            i += 2
        else:
            if ipa[i] in _PHONES_1:
                phones.append(ipa[i])
            i += 1
    return ' '.join(phones)


paired_ds = paired_ds.map(
    lambda item: {'English ASR transcriptions': _space_separate(item['English ASR transcriptions'])},
    desc='Normalising synthetic IPA',
)
print('Before:', 'ʌɾəbʌd')
print('After: ', _space_separate('ʌɾəbʌd'))

Normalising synthetic IPA: 100%|██████████| 19136/19136 [00:02<00:00, 8797.61 examples/s] 

Before: ʌɾəbʌd
After:  ʌ ɾ ə b ʌ d


In [9]:
paired_ds[0]

{'audio': {'path': 'carbad.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(13440,)),
  'sampling_rate': 16000},
 'phonetic': 'k a ɾ ə b ə d',
 'English ASR transcriptions': 'ʌ ɾ ə b ʌ d',
 'synthetic_audio_path': {'path': '0.mp3',
  'array': array([ 3.18323146e-12,  2.27373675e-12, -9.54969437e-12, ...,
          8.29194323e-06, -2.34242179e-06, -2.22762756e-05], shape=(11136,)),
  'sampling_rate': 16000}}

In [10]:
l2_ds = build_l2_training_set(paired_ds)

Building L2 training set: 100%|██████████| 19136/19136 [08:24<00:00, 37.91 examples/s] 


In [11]:
print(l2_ds)
print('\nSample IPA:', l2_ds[0]['ipa'])

Dataset({
    features: ['audio', 'ipa', 'order'],
    num_rows: 19136
})

Sample IPA: ʌ ɾ ə b ʌ d k a ɾ ə b ə d


# Split dataset
.8/.1/.1 split to be comparable to other models

In [12]:
paired_ds

Dataset({
    features: ['audio', 'phonetic', 'English ASR transcriptions', 'synthetic_audio_path'],
    num_rows: 19136
})

In [13]:
ds_temp = l2_ds.train_test_split(test_size=0.2, seed=RANDOM_SEED)
ds_final = ds_temp["test"].train_test_split(test_size=0.5, seed=RANDOM_SEED)

final_ds = DatasetDict({
    "train":      ds_temp["train"],
    "validation": ds_final["train"],
    "test":       ds_final["test"],
})

In [14]:
final_ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 15308
    })
    validation: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
    test: Dataset({
        features: ['audio', 'ipa', 'order'],
        num_rows: 1914
    })
})

# Save datasets

In [15]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
final_ds.save_to_disk(str(OUTPUT_PATH))
print(f'Saved to {OUTPUT_PATH}')


Saving the dataset (1/1 shards): 100%|██████████| 1914/1914 [00:00<00:00, 4044.58 examples/s]

Saved to /home/peter/Desktop/thesis/ThesisProject/data/synthetic/l2_training_set


# Build IPA vocab

In [16]:
train_phonetics = [phone for x in final_ds["train"] for phone in x['ipa'].split()]
valid_phonetics = [phone for x in final_ds["validation"] for phone in x['ipa'].split()]
test_phonetics = [phone for x in final_ds["test"] for phone in x['ipa'].split()]

print("num of train phones:\t", len(set(train_phonetics)))
print("num of valid phones:\t", len(set(valid_phonetics)))
print("num of test phones:\t", len(set(test_phonetics)))

num of train phones:	 87
num of valid phones:	 84
num of test phones:	 83


In [17]:
vocab_train = list(set(train_phonetics)) + [' ']
vocab_valid = list(set(valid_phonetics)) + [' ']
vocab_test  = list(set(test_phonetics)) + [' ']

In [18]:
vocab_list = list(set(vocab_train + vocab_valid + vocab_test))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

print(vocab_dict)

{' ': 0, 'a': 1, 'ai': 2, 'au': 3, 'aɪ': 4, 'aʊ': 5, 'aː': 6, 'b': 7, 'bʲ': 8, 'c': 9, 'd': 10, 'dʲ': 11, 'e': 12, 'eɪ': 13, 'eː': 14, 'f': 15, 'fʲ': 16, 'g': 17, 'h': 18, 'hʲ': 19, 'i': 20, 'ia': 21, 'iː': 22, 'iˑə': 23, 'j': 24, 'k': 25, 'l': 26, 'lʲ': 27, 'l̊': 28, 'l̩': 29, 'm': 30, 'mʲ': 31, 'n': 32, 'nʲ': 33, 'n̊': 34, 'n̩': 35, 'o': 36, 'oʊ': 37, 'oː': 38, 'p': 39, 'pʲ': 40, 's': 41, 't': 42, 'tʲ': 43, 'u': 44, 'ua': 45, 'uː': 46, 'uˑə': 47, 'v': 48, 'vʲ': 49, 'w': 50, 'x': 51, 'z': 52, 'zʲ': 53, 'æ': 54, 'ç': 55, 'ð': 56, 'ŋ': 57, 'ɑ': 58, 'ɒ': 59, 'ɔ': 60, 'ɔɪ': 61, 'ə': 62, 'ə̥': 63, 'ɚ': 64, 'ɛ': 65, 'ɝ': 66, 'ɟ': 67, 'ɡ': 68, 'ɣ': 69, 'ɦ': 70, 'ɨ': 71, 'ɪ': 72, 'ɲ': 73, 'ɲ̊': 74, 'ɹ': 75, 'ɾ': 76, 'ɾʲ': 77, 'ɾ̃': 78, 'ʃ': 79, 'ʉ': 80, 'ʊ': 81, 'ʌ': 82, 'ʒ': 83, 'ʔ': 84, 'ʤ': 85, 'ʧ': 86, 'θ': 87}


In [19]:
# make the space more intuitive to understand
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

90

In [20]:
# save vocab.json
import json
with open(str(OUTPUT_PATH.parent)+"/vocab.json", 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)